In [71]:
from dotenv import load_dotenv
load_dotenv()

True

In [72]:
from langgraph.graph import START,END,StateGraph
from langchain_groq import ChatGroq
from pydantic import BaseModel
from langgraph.types import interrupt,Command
from langgraph.checkpoint.memory import InMemorySaver
from typing import Literal

In [73]:
llm=ChatGroq(model="openai/gpt-oss-20b")

In [83]:
class MailState(BaseModel):
    query:str=""
    draft:str=""
    human_feedback:str=""
    final_res:str=""
    

In [121]:
### defining the nodes
def draft_mail_node(state:MailState)->MailState:
    if state.human_feedback:
        draft=llm.invoke(f"""
            you have generated this mail :{state.draft}
            human feedback to apply :{state.human_feedback}
            rewrite this draft email for {state.query}
        """).content
    else:
        draft=llm.invoke(state.query).content
    state.draft=draft
    return state

def human_feedback_node(state:MailState)->MailState:
    feedback=interrupt({
        "draft_email":state.draft,
        "question":"Do you want ot continue or rewrite the mail"
    })
    fb=(feedback or  "").strip().lower()
    if fb in ("approved","ok","yes"):
        state.human_feedback=""
        return state
    else:
        state.human_feedback=feedback
        return state

def finalize_node(state:MailState)->MailState:
    print("email sent successfully")
    state.final_res=state.draft
    return state

def condition_routing(state:MailState)->Literal["draft","finalize"]:
    if state.human_feedback:
        return "draft"
    return "finalize"


In [122]:
graph_builder=StateGraph(MailState)
graph_builder.add_node("draft",draft_mail_node)
graph_builder.add_node("human_feedback",human_feedback_node)
graph_builder.add_node("finalize",finalize_node)

graph_builder.add_edge(START,"draft")
graph_builder.add_edge("draft","human_feedback")
graph_builder.add_conditional_edges("human_feedback",condition_routing)
graph_builder.add_edge("finalize",END)

graph=graph_builder.compile(checkpointer=InMemorySaver())

In [123]:
config={"configurable":{"thread_id":"1"}}                     

In [124]:
res=graph.invoke({"query":"i want to send a email for medical leave request"}, config=config)

In [131]:
print(res["draft"])
res

**Subject:** Medical Leave Request – [Your Full Name]  

---

Dear [Supervisor’s Name],

I am writing to request medical leave from **4 June to 7 June** due to a medical condition that requires treatment and recovery time. I have attached the doctor’s note confirming my need for this leave.

During my absence, I will ensure a smooth handover of my current responsibilities. I have briefed [Colleague’s Name] on the status of my key tasks and provided them with the necessary documentation. If you need any additional information or if there are forms I should complete, please let me know.

Thank you for your understanding and support. I will keep you updated on my health and the expected return date.

Thank you,

**[Your Full Name]**  
**[Your Job Title]**  
**[Department]**  
**[Phone Number]**  
**[Email Address]**


{'query': 'i want to send a email for medical leave request',
 'draft': '**Subject:** Medical Leave Request –\u202f[Your Full Name]  \n\n---\n\nDear [Supervisor’s Name],\n\nI am writing to request medical leave from **4\u202fJune to 7\u202fJune** due to a medical condition that requires treatment and recovery time. I have attached the doctor’s note confirming my need for this leave.\n\nDuring my absence, I will ensure a smooth handover of my current responsibilities. I have briefed [Colleague’s Name] on the status of my key tasks and provided them with the necessary documentation. If you need any additional information or if there are forms I should complete, please let me know.\n\nThank you for your understanding and support. I will keep you updated on my health and the expected return date.\n\nThank you,\n\n**[Your Full Name]**  \n**[Your Job Title]**  \n**[Department]**  \n**[Phone Number]**  \n**[Email Address]**',
 'human_feedback': '',
 'final_res': '**Subject:** Medical Leave Re

In [126]:
res=graph.invoke(Command(resume="i want to change the leave dates and new dates are 4th june to 7yh june and i want to remove the place holders and just say thankyou"),config=config)

In [127]:
print(res["draft"])
res

**Subject:** Medical Leave Request – [Your Full Name]  

---

Dear [Supervisor’s Name],

I am writing to request medical leave from **4 June to 7 June** due to a medical condition that requires treatment and recovery time. I have attached the doctor’s note confirming my need for this leave.

During my absence, I will ensure a smooth handover of my current responsibilities. I have briefed [Colleague’s Name] on the status of my key tasks and provided them with the necessary documentation. If you need any additional information or if there are forms I should complete, please let me know.

Thank you for your understanding and support. I will keep you updated on my health and the expected return date.

Thank you,

**[Your Full Name]**  
**[Your Job Title]**  
**[Department]**  
**[Phone Number]**  
**[Email Address]**


{'query': 'i want to send a email for medical leave request',
 'draft': '**Subject:** Medical Leave Request –\u202f[Your Full Name]  \n\n---\n\nDear [Supervisor’s Name],\n\nI am writing to request medical leave from **4\u202fJune to 7\u202fJune** due to a medical condition that requires treatment and recovery time. I have attached the doctor’s note confirming my need for this leave.\n\nDuring my absence, I will ensure a smooth handover of my current responsibilities. I have briefed [Colleague’s Name] on the status of my key tasks and provided them with the necessary documentation. If you need any additional information or if there are forms I should complete, please let me know.\n\nThank you for your understanding and support. I will keep you updated on my health and the expected return date.\n\nThank you,\n\n**[Your Full Name]**  \n**[Your Job Title]**  \n**[Department]**  \n**[Phone Number]**  \n**[Email Address]**',
 'human_feedback': 'i want to change the leave dates and new dates 

In [128]:
res

{'query': 'i want to send a email for medical leave request',
 'draft': '**Subject:** Medical Leave Request –\u202f[Your Full Name]  \n\n---\n\nDear [Supervisor’s Name],\n\nI am writing to request medical leave from **4\u202fJune to 7\u202fJune** due to a medical condition that requires treatment and recovery time. I have attached the doctor’s note confirming my need for this leave.\n\nDuring my absence, I will ensure a smooth handover of my current responsibilities. I have briefed [Colleague’s Name] on the status of my key tasks and provided them with the necessary documentation. If you need any additional information or if there are forms I should complete, please let me know.\n\nThank you for your understanding and support. I will keep you updated on my health and the expected return date.\n\nThank you,\n\n**[Your Full Name]**  \n**[Your Job Title]**  \n**[Department]**  \n**[Phone Number]**  \n**[Email Address]**',
 'human_feedback': 'i want to change the leave dates and new dates 

In [129]:
res=graph.invoke(Command(resume="yes"),config=config)

email sent successfully


In [130]:
res

{'query': 'i want to send a email for medical leave request',
 'draft': '**Subject:** Medical Leave Request –\u202f[Your Full Name]  \n\n---\n\nDear [Supervisor’s Name],\n\nI am writing to request medical leave from **4\u202fJune to 7\u202fJune** due to a medical condition that requires treatment and recovery time. I have attached the doctor’s note confirming my need for this leave.\n\nDuring my absence, I will ensure a smooth handover of my current responsibilities. I have briefed [Colleague’s Name] on the status of my key tasks and provided them with the necessary documentation. If you need any additional information or if there are forms I should complete, please let me know.\n\nThank you for your understanding and support. I will keep you updated on my health and the expected return date.\n\nThank you,\n\n**[Your Full Name]**  \n**[Your Job Title]**  \n**[Department]**  \n**[Phone Number]**  \n**[Email Address]**',
 'human_feedback': '',
 'final_res': '**Subject:** Medical Leave Re